In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mnLa683Yzcd07PEOtSdC")
project = rf.workspace("blue-halo").project("vegetation-segmentation")
version = project.version(4)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 35.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to vegetation-segmentation-4 in yolov8:: 100%|██████████| 2148/2148 [00:00<00:00, 2443.18it/s]


In [ ]:
import os
import random
import numpy as np
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
# 1. PATH TO VEGETATION DATASET
VEG_DATASET = "vegetation-segmentation-4"

def find_images(folder):
    """Return all image paths in folder."""
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(root, f))
    return paths
print(" Scanning vegetation dataset...")
images = find_images(VEG_DATASET)
print(f"Total vegetation images found: {len(images)}")
if len(images) == 0:
    print(" No vegetation images found. Check the folder name.")
    exit()
# 2. OUTPUT FOLDERS
os.makedirs("processed/train", exist_ok=True)
os.makedirs("processed/test", exist_ok=True)
os.makedirs("processed/augmented", exist_ok=True)
os.makedirs("processed/encoded_pixels", exist_ok=True)
#PREPROCESSING + AUGMENTATION
resize = transforms.Resize((512, 512))

augment = transforms.Compose([
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((512, 512))
])
#TRAIN–TEST SPLIT
random.shuffle(images)
split = int(0.8 * len(images))
train = images[:split]
test = images[split:]
print(f"Training images: {len(train)}")
print(f"Testing images: {len(test)}")
#FUNCTION TO PROCESS IMAGE
def process_image(img_path, save_folder, do_aug=False):
    img = Image.open(img_path).convert("RGB")
    resized = resize(img)
    resized.save(os.path.join(save_folder, os.path.basename(img_path)))
    if do_aug:
        aug_img = augment(img)
        aug_img.save(f"processed/augmented/AUG_{os.path.basename(img_path)}")
    arr = np.array(resized)
    h, w, _ = arr.shape
    center_pixel = arr[h//2, w//2]
    pixel_file = f"processed/encoded_pixels/{os.path.basename(img_path)}.txt"
    np.savetxt(pixel_file, center_pixel.reshape(1, 3), fmt='%d')
print("\n Processing training images...")
for img in tqdm(train):
    process_image(img, "processed/train", do_aug=True)
print("\n Processing test images...")
for img in tqdm(test):
    process_image(img, "processed/test", do_aug=False)
print("\n PREPROCESSING COMPLETE (VEGETATION ONLY)")
print(" Processed images → processed/train , processed/test")
print(" Augmented images → processed/augmented")
print(" Encoded RGB pixels → processed/encoded_pixels")

 Scanning vegetation dataset...
Total vegetation images found: 1068
Training images: 854
Testing images: 214

 Processing training images...


100%|██████████| 854/854 [01:02<00:00, 13.57it/s]



 Processing test images...


100%|██████████| 214/214 [00:03<00:00, 55.54it/s]


 PREPROCESSING COMPLETE (VEGETATION ONLY)
 Processed images → processed/train , processed/test
 Augmented images → processed/augmented
 Encoded RGB pixels → processed/encoded_pixels


In [ ]:
train_label_files = os.listdir("/content/processed/train")
test_label_files = os.listdir("/content/processed/test")

print("Train labels:", len(train_label_files))
print("Test labels:", len(test_label_files))


Train labels: 854
Test labels: 214


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.7 MB/s eta 0:00:00


In [ ]:
!yolo task=segment mode=train model=yolov8n-seg.pt data="/content/vegetation-segmentation-4/data.yaml" epochs=50 imgsz=512 batch=16


WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n-seg.pt... HTTP Error 503: Service Unavailable
######################################################################## 100.0%
Ultralytics 8.3.236 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetation-segmentation-4/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

IMAGE_DIR = "/content/processed/train"   # Your dataset folder in Colab
OUTPUT_CSV = "/content/veg_features.csv" # Output CSV in Colab

IMG_SIZE = 256
rows = []

# Get image files
image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for file in tqdm(image_files, desc="Processing images"):
    img_path = os.path.join(IMAGE_DIR, file)
    img = cv2.imread(img_path)

    if img is None:
        print("❌ Skipped invalid image:", file)
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Correct RGB values (cv2 = BGR)
    B_mean = img[:, :, 0].mean()
    G_mean = img[:, :, 1].mean()
    R_mean = img[:, :, 2].mean()

    # HSV conversion
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    H_mean = hsv[:, :, 0].mean()
    S_mean = hsv[:, :, 1].mean()
    V_mean = hsv[:, :, 2].mean()

    # NDVI calculation
    NDVI = (G_mean - R_mean) / (G_mean + R_mean + 1e-6)

    rows.append([file, R_mean, G_mean, B_mean, H_mean, S_mean, V_mean, NDVI])

# Save CSV
df = pd.DataFrame(rows, columns=["Image", "R", "G", "B", "H", "S", "V", "NDVI"])
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Features saved successfully!")
print("📄 CSV Path:", OUTPUT_CSV)
print("📌 Total images processed:", len(df))


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# -------------------------
# PATHS
# -------------------------
IMAGE_DIR = "/content/vegetation-segmentation-4/test/images"   # change if needed
OUTPUT_CSV = "/content/veg_features.csv"

# -------------------------
# FUNCTION: Extract Features
# -------------------------
def extract_features(img_path):
    img = cv2.imread(img_path)

    if img is None:
        return None

    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Mean colors
    mean_r = np.mean(img[:, :, 2])
    mean_g = np.mean(img[:, :, 1])
    mean_b = np.mean(img[:, :, 0])

    # Texture Features (GLCM)
    glcm = graycomatrix(gray, distances=[1], angles=[0], symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')[0][0]
    dissimilarity = graycoprops(glcm, 'dissimilarity')[0][0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0][0]
    energy = graycoprops(glcm, 'energy')[0][0]
    correlation = graycoprops(glcm, 'correlation')[0][0]

    return [mean_r, mean_g, mean_b, contrast, dissimilarity, homogeneity, energy, correlation]


# -------------------------
# PROCESS ALL IMAGES
# -------------------------
data = []
filenames = []

for fname in os.listdir(IMAGE_DIR):
    if fname.lower().endswith((".jpg", ".jpeg", ".png")):
        fpath = os.path.join(IMAGE_DIR, fname)

        feats = extract_features(fpath)

        if feats:
            data.append(feats)
            filenames.append(fname)

# -------------------------
# SAVE TO features.csv
# -------------------------
df = pd.DataFrame(data, columns=[
    "mean_R", "mean_G", "mean_B",
    "contrast", "dissimilarity", "homogeneity",
    "energy", "correlation"
])

df.insert(0, "filename", filenames)

df.to_csv(OUTPUT_CSV, index=False)

print("✅ Saved successfully:", OUTPUT_CSV)
print(df.head())


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ------------------------------
# 1. Load features
# ------------------------------
df = pd.read_csv("/content/veg_features.csv")

# Create label (simple vegetation rule)
df['label'] = (df['mean_G'] > df['mean_R']).astype(int)

# Features & labels
X = df[['mean_R','mean_G','mean_B','contrast','dissimilarity','homogeneity','energy','correlation']]
y = df['label']

# ------------------------------
# 2. Train-test split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling for LR and SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------
# 3. Train models
# ------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel="rbf"),
    "Random Forest": RandomForestClassifier(n_estimators=300),
    "XGBoost": XGBClassifier(eval_metric="logloss")
}

accuracies = {}

for name, model in models.items():
    if name in ["Random Forest", "XGBoost"]:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = acc

# ------------------------------
# 4. Print all accuracies
# ------------------------------
print("\nMODEL ACCURACIES:")
for name, acc in accuracies.items():
    print(f"{name}: {acc*100:.2f}%")

# ------------------------------
# 5. Print best model
# ------------------------------
best_model = max(accuracies, key=accuracies.get)
best_accuracy = accuracies[best_model]


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.bar(accuracies.keys(), accuracies.values())
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=30)
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ------------------------------
# 1. Load features
# ------------------------------
df = pd.read_csv("/content/veg_features.csv")

# Create label (vegetation vs non-vegetation)
df['label'] = (df['mean_G'] > df['mean_R']).astype(int)

# Features & labels
X = df[['mean_R','mean_G','mean_B','contrast','dissimilarity','homogeneity','energy','correlation']]
y = df['label']

# ------------------------------
# 2. Train-test split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------
# 3. Train SVM
# ------------------------------
svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_train_scaled, y_train)

# ------------------------------
# 4. Evaluate
# ------------------------------
y_pred = svm_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ SVM Accuracy: {accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ------------------------------
# 5. Optional: Save the trained model
# ------------------------------
joblib.dump(svm_model, "/content/svm_veg_model.joblib")
joblib.dump(scaler, "/content/scaler.joblib")
print("✅ SVM model saved as: /content/svm_veg_model.joblib")


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Predict on test data
y_pred = svm_model.predict(X_test_scaled)

# 2. Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ SVM Accuracy: {accuracy*100:.2f}%")

# 3. Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 4. Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Veg", "Veg"], yticklabels=["Non-Veg", "Veg"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# Paths
TEST_DIR = "/content/vegetation-segmentation-4/test/images"  # change if needed

# Function to extract same features as training
def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None

    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Mean colors
    mean_r = np.mean(img[:, :, 2])
    mean_g = np.mean(img[:, :, 1])
    mean_b = np.mean(img[:, :, 0])

    # Texture features (GLCM)
    from skimage.feature import graycomatrix, graycoprops
    glcm = graycomatrix(gray, distances=[1], angles=[0], symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0,0]
    dissimilarity = graycoprops(glcm, 'dissimilarity')[0,0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0,0]
    energy = graycoprops(glcm, 'energy')[0,0]
    correlation = graycoprops(glcm, 'correlation')[0,0]

    return [mean_r, mean_g, mean_b, contrast, dissimilarity, homogeneity, energy, correlation]

# Collect features for all test images
test_data = []
test_files = []

for f in os.listdir(TEST_DIR):
    if f.lower().endswith((".jpg", ".jpeg", ".png")):
        feats = extract_features(os.path.join(TEST_DIR, f))
        if feats:
            test_data.append(feats)
            test_files.append(f)

# Convert to DataFrame
X_test_df = pd.DataFrame(test_data, columns=['mean_R','mean_G','mean_B','contrast','dissimilarity','homogeneity','energy','correlation'])

# Scale features (same scaler used for training)
X_test_scaled = scaler.transform(X_test_df)

# Predict with trained SVM
y_pred_test = svm_model.predict(X_test_scaled)

# Show results
results_df = pd.DataFrame({
    "Image": test_files,
    "Prediction": y_pred_test
})
results_df['Prediction'] = results_df['Prediction'].map({0:"Non-Vegetation", 1:"Vegetation"})

print(results_df)


In [ ]:
# Count predictions
veg_count = (y_pred_test == 1).sum()
nonveg_count = (y_pred_test == 0).sum()
total = len(y_pred_test)

# Calculate percentages
veg_percent = (veg_count / total) * 100
nonveg_percent = (nonveg_count / total) * 100

print(f"Total images: {total}")
print(f"Vegetation: {veg_count} ({veg_percent:.2f}%)")
print(f"Non-Vegetation: {nonveg_count} ({nonveg_percent:.2f}%)")


In [ ]:
from google.colab import files

uploaded = files.upload()  # This opens a file picker


In [ ]:
img_path = list(uploaded.keys())[0]  # Get the uploaded file name
print("Uploaded image path:", img_path)


In [ ]:
import cv2
import numpy as np
from joblib import load

# Load model and scaler
svm_model = load("/content/svm_veg_model.joblib")
scaler = load("/content/scaler.joblib")

# Feature extraction function
def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print("❌ Could not read image!")
        return None

    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Example features (same as used in training)
    mean_r = np.mean(img[:, :, 2])
    mean_g = np.mean(img[:, :, 1])
    mean_b = np.mean(img[:, :, 0])


    # For simplicity, we skip GLCM here (or you can include it if your model uses it)
    feats = np.array([[mean_r, mean_g, mean_b, 0, 0, 0, 0, 0]])  # replace 0s with actual texture features
    feats_scaled = scaler.transform(feats)
    pred = svm_model.predict(feats_scaled)[0]
    return "Vegetation" if pred == 1 else "Non-Vegetation"

# Predict
result = extract_features(img_path)
print("Prediction:", result)
